In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from tqdm import tqdm
from alerce.core import Alerce

In [2]:
# better than rbf for supernovae?
def matern32(t1, t2, length_scale, amplitude):
    r = np.abs(t1[:, None] - t2[None, :]) / length_scale
    return amplitude**2 * (1 + np.sqrt(3) * r) * np.exp(-np.sqrt(3) * r)

# most likely hetero gp
def log_noise_gp(t, log_var_obs, length_scale=0.3, amplitude=1.0, noise_floor=1e-4):
    K = matern32(t, t, length_scale, amplitude)
    K += noise_floor * np.eye(len(t))
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(t)))
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, log_var_obs))    
    return K, L, alpha

In [3]:
class HeteroscedasticGP:
    """
    Use EM algo to approximate noise levels (this is just most likely hetero)
    """
    def __init__(self, n_iter=3):
        self.n_iter = n_iter

    def fit(self, t, y, init_params=None):
        t = np.asarray(t, dtype=float)
        y = np.asarray(y, dtype=float)
        
        init_ls = init_params[0] if init_params else 0.2
        init_amp = init_params[1] if init_params else 1.0
        init_noise = init_params[2] if init_params else 0.05
        
        noise_var = np.full(len(t), init_noise**2)
        
        for _ in range(self.n_iter):
            def neg_log_marginal(params):
                ls, amp = np.exp(params[0]), np.exp(params[1])
                K = matern32(t, t, ls, amp)
                Ky = K + np.diag(noise_var)
                try:
                    L = np.linalg.cholesky(Ky)
                except np.linalg.LinAlgError:
                    return 1e10
                alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
                return (0.5 * y @ alpha
                        + np.sum(np.log(np.diag(L)))
                        + 0.5 * len(t) * np.log(2 * np.pi))

            # bfgs = goat optimizer
            res = minimize(neg_log_marginal, [np.log(init_ls), np.log(init_amp)],
                           method='L-BFGS-B',
                           bounds=[(-4, 1), (-3, 2)])
            ls, amp = np.exp(res.x[0]), np.exp(res.x[1])
            
            K = matern32(t, t, ls, amp)
            Ky = K + np.diag(noise_var)
            L = np.linalg.cholesky(Ky + 1e-8 * np.eye(len(t)))
            alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
            mu = K @ alpha
            residuals = y - mu
            
            log_sq_resid = np.log(residuals**2 + 1e-6)
            _, _, noise_alpha = log_noise_gp(t, log_sq_resid)
            noise_K = matern32(t, t, 0.15, 1.0)
            noise_mu = noise_K @ noise_alpha
            noise_var = np.exp(noise_mu) + 1e-6
        
        self.ls_ = ls
        self.amp_ = amp
        self.t_train_ = t
        self.y_train_ = y
        self.noise_var_ = noise_var
        self.L_ = L
        self.alpha_ = alpha
        self.K_train_ = K
        return self

    def predict(self, t_test):
        t_test = np.asarray(t_test, dtype=float)
        K_s = matern32(t_test, self.t_train_, self.ls_, self.amp_)
        K_ss = matern32(t_test, t_test, self.ls_, self.amp_)
        
        mu = K_s @ self.alpha_
        v = np.linalg.solve(self.L_, K_s.T)
        var = np.diag(K_ss) - np.sum(v**2, axis=0)
        
        noise_K = matern32(t_test, self.t_train_, 0.15, 1.0)
        log_sq_resid = np.log(self.noise_var_)
        _, _, noise_alpha = log_noise_gp(self.t_train_, log_sq_resid)
        noise_mu = noise_K @ noise_alpha
        noise_var_test = np.exp(noise_mu) + 1e-6
        
        return mu, np.sqrt(np.maximum(var, 0)), np.sqrt(noise_var_test)

    def log_likelihood(self, t_new, y_new):
        """Log p(y_new | GP posterior) — lower = more anomalous"""
        mu, sig_gp, sig_noise = self.predict(t_new)
        total_var = sig_gp**2 + sig_noise**2
        return -0.5 * np.sum((y_new - mu)**2 / total_var + np.log(total_var))

In [4]:
def load_curves(csv_dir, min_pts=20, max_pts=250):
    """
    Same preprocessing as in DBSCAN.ipynb.
    """
    curves_data = []  
    files = os.listdir(csv_dir)
    for fname in tqdm(files, desc="Processing curves"):
        oid = fname.replace(".csv", "")
        df = pd.read_csv(os.path.join(csv_dir, fname))
        g = df[df['fid'] == 1]
        r = df[df['fid'] == 2].sort_values('mjd')
        
        if len(r) < min_pts or len(g) < min_pts or len(r) > max_pts:
            continue
        
        color_offset = g['magpsf'].median() - r['magpsf'].median()
        mag = r['magpsf'].values + color_offset
        t = r['mjd'].values
        
        if np.any(np.isnan(mag)) or np.any(np.isnan(t)):
            continue
        
        # Normalize
        t_norm = (t - t.min()) / (t.max() - t.min() + 1e-9)
        mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-9)
        
        curves_data.append((oid, t_norm, mag_norm, t, mag))
    
    return curves_data

In [5]:
def fit_all_gps(curves_data):
    fitted = []
    for oid, t, y, raw_t, raw_y in tqdm(curves_data, desc="Fitting GPs"):
        try:
            gp = HeteroscedasticGP(n_iter=3).fit(t, y)
            fitted.append((oid, gp, t, y))
        except Exception as e:
            tqdm.write(f"Skipping {oid}: {e}") 
    return fitted

def build_template(fitted_gps, n_grid=200):
    """
    Evaluate all fitted GPs on a common time grid.
    Template = mean of individual posterior means.
    Template uncertainty = std across curves.
    """
    t_grid = np.linspace(0, 1, n_grid)
    all_means = []
    all_noise_vars = []
    
    for oid, gp, t, y in fitted_gps:
        mu, sig_gp, sig_noise = gp.predict(t_grid)
        all_means.append(mu)
        all_noise_vars.append(sig_noise**2)
    
    all_means = np.array(all_means)      # (n_curves, n_grid)
    all_noise = np.array(all_noise_vars)  # (n_curves, n_grid)
    
    template_mu = np.mean(all_means, axis=0)
    template_std = np.std(all_means, axis=0)  # population spread
    mean_noise = np.mean(all_noise, axis=0)   # avg heteroscedastic noise
    
    return t_grid, template_mu, template_std, mean_noise, all_means, all_noise

In [6]:
def score_precursor(gp, t_obs, y_obs, t_grid, template_mu, template_std,
                    pre_peak_frac=0.3, sigma_thresh=2.5):
    """
    Check window before max peak, find anomalies there by comparing anomaly gp to total gp distro.
    """
    mu, sig_gp, sig_noise = gp.predict(t_grid)
    
    peak_idx = np.argmin(mu)
    peak_t = t_grid[peak_idx]
    
    pre_mask = t_grid < peak_t * pre_peak_frac
    if pre_mask.sum() < 3:
        pre_mask = t_grid < 0.15  # fallback: first 15%
    
    deviation = np.abs(mu[pre_mask] - template_mu[pre_mask])
    normalized_dev = deviation / (template_std[pre_mask] + 1e-6)
    mean_deviation_score = np.mean(normalized_dev)
    
    pre_noise = sig_noise[pre_mask].mean()
    post_mask = t_grid > peak_t * 1.2
    post_noise = sig_noise[post_mask].mean() if post_mask.sum() > 3 else sig_noise.mean()
    noise_excess = pre_noise / (post_noise + 1e-6)
    
    # log-likelihood of observed pre-peak points under template
    pre_obs_mask = t_obs < peak_t * pre_peak_frac
    if pre_obs_mask.sum() >= 2:
        t_pre = t_obs[pre_obs_mask]
        y_pre = y_obs[pre_obs_mask]
        tmpl_mu_obs = np.interp(t_pre, t_grid, template_mu)
        tmpl_std_obs = np.interp(t_pre, t_grid, template_std)
        ll_score = -np.mean((y_pre - tmpl_mu_obs)**2 / (tmpl_std_obs**2 + 1e-6))
    else:
        ll_score = 0.0
    
    is_precursor = (mean_deviation_score > sigma_thresh) or (noise_excess > 2.0)
    
    return {
        'peak_t': peak_t,
        'mean_deviation_score': mean_deviation_score,
        'noise_excess_ratio': noise_excess,
        'pre_peak_ll': ll_score,
        'is_precursor': is_precursor,
    }


In [7]:
csv_dir = "csv_files"

print("=" * 50)
curves_data = load_curves(csv_dir)
print(f"Loaded {len(curves_data)} curves")

print("=" * 50)

# takes VERY long time. needs to run most likely for each and every curve
fitted_gps = fit_all_gps(curves_data)
print(f"Fitted {len(fitted_gps)} GPs")

print("=" * 50)

print("Building population template...")
t_grid, template_mu, template_std, mean_noise, all_means, all_noise = build_template(fitted_gps)

print("=" * 50)

print("Scoring for precursor activity...")
results = []
for oid, gp, t, y in fitted_gps:
    scores = score_precursor(gp, t, y, t_grid, template_mu, template_std)
    scores['oid'] = oid
    results.append(scores)

results_df = pd.DataFrame(results).sort_values('mean_deviation_score', ascending=False)
precursors = results_df[results_df['is_precursor']]
print(f"\nPrecursor candidates: {len(precursors)} / {len(results_df)}")
print(precursors[['oid', 'mean_deviation_score', 'noise_excess_ratio', 'pre_peak_ll']].head(10))

Processing curves: 100%|██████████| 12258/12258 [01:54<00:00, 106.84it/s]


Loaded 2869 curves


Fitting GPs: 100%|██████████| 2869/2869 [05:10<00:00,  9.24it/s]
/tmp/ipykernel_1455578/3077978115.py:74: RuntimeWarning: overflow encountered in exp
  noise_var_test = np.exp(noise_mu) + 1e-6


Fitted 2869 GPs
Building population template...
Scoring for precursor activity...


/tmp/ipykernel_1455578/2343895166.py:22: RuntimeWarning: invalid value encountered in scalar divide
  noise_excess = pre_noise / (post_noise + 1e-6)



Precursor candidates: 725 / 2869
               oid  mean_deviation_score  noise_excess_ratio  pre_peak_ll
2344  ZTF18abiinxx             21.191806       2.619607e-120    -0.541547
1746  ZTF22aafmixt              7.276438        9.221496e-62    -0.213104
735   ZTF21aagyzkb              7.077614        2.703063e-67    -0.070854
830   ZTF21acbmqmy              6.426142        1.108929e+29    -0.022244
2319  ZTF18adjzuer              4.144774        0.000000e+00    -0.499659
1467  ZTF18abmqvsn              3.928094        5.462481e-74    -0.537925
2735  ZTF22aafrblg              3.882719        5.430097e-21    -0.067573
210   ZTF18abmszya              3.758889        4.159392e-80    -0.071388
1838  ZTF26aajvkwq              3.287356        5.940513e-21    -0.372988
2     ZTF24abmyoit              3.272498        3.930831e-71    -0.015925


In [8]:
def predict_new(oid, t_grid, template_mu, template_std, sigma_thresh=2.5):
    alerce = Alerce()
    new_df = alerce.query_detections(oid, format="pandas")
    new_r = new_df[new_df['fid'] == 2].sort_values('mjd').drop_duplicates('mjd')
    new_g = new_df[new_df['fid'] == 1]
    
    color_offset = new_g['magpsf'].median() - new_r['magpsf'].median()
    mag = new_r['magpsf'].values + color_offset
    t = new_r['mjd'].values
    
    t_norm = (t - t.min()) / (t.max() - t.min() + 1e-9)
    mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-9)
    
    gp = HeteroscedasticGP(n_iter=3).fit(t_norm, mag_norm)
    scores = score_precursor(gp, t_norm, mag_norm, t_grid, template_mu, template_std,
                             sigma_thresh=sigma_thresh)
    
    return scores['is_precursor'], scores

In [9]:
oid = "ZTF22aafmixt"
predict_new(oid, t_grid, template_mu, template_std)

(np.True_,
 {'peak_t': np.float64(0.06030150753768844),
  'mean_deviation_score': np.float64(7.2764385147023365),
  'noise_excess_ratio': np.float64(9.222153253132706e-62),
  'pre_peak_ll': np.float64(-0.2131037807264434),
  'is_precursor': np.True_})

In [10]:
file = "snii_with_precursors.csv"
df = pd.read_csv(file)
df = df[df['bands'] == 'R,g']
oids = df['ztf_oid']
for oid in oids:
    try:
        result, _ = predict_new(oid, t_grid, template_mu, template_std)
        print(f"{oid}: {result}")
    except Exception as e:
        print("didn't work")

ZTF19aaimlnn: False
ZTF20abolrpg: False
ZTF18acealtv: False
ZTF25abxlcab: True
ZTF20acxzmro: False
ZTF25acfwips: True
ZTF20aavuiwf: False
ZTF21abifctk: False
ZTF20acxztft: True
ZTF20aadyeye: False
ZTF19actlcbw: False
ZTF20acwhxah: True
ZTF19aanwfqd: False
ZTF20aafnsco: False
ZTF18acdwwql: False
ZTF19adnuazm: False


/tmp/ipykernel_1455578/3077978115.py:74: RuntimeWarning: overflow encountered in exp
  noise_var_test = np.exp(noise_mu) + 1e-6


ZTF19aargopz: False
ZTF19acvwoty: False
ZTF19abkgtka: False
ZTF18adrvopa: False
ZTF22abwtwmr: True
ZTF23absmvjg: True
ZTF19acnpwqg: True
ZTF21aadrtze: True
ZTF21abdrwiv: True
ZTF19aaiohcc: False
ZTF21acjmutc: False
ZTF23absdnkt: False
ZTF19aapfqce: True
ZTF18aazyhkf: False
ZTF20abapeub: False
ZTF19aaoiawb: True
ZTF23aablfwp: False
ZTF22abnfbik: False
